# Aula 10 | Melhorar RAG sem perder rigor

**Inteligência Artificial para Decisão** · Macroeconomia e Finanças · notebook local.

**Pergunta:** Quando aumentar o número de trechos recuperados melhora a resposta e quando apenas acrescenta ruído?

**Procedimento:** execute as células, confira os resultados e modifique uma hipótese. Não há chamadas a serviços externos. Todos os dados são sintéticos ou fictícios.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Localiza a raiz do curso, inclusive ao abrir o notebook a partir da pasta da aula.
def achar_raiz():
    origem = Path.cwd().resolve()
    for pasta in (origem, *origem.parents):
        if (pasta / "dados" / "README.md").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Abra o notebook dentro do repositório extraído, preservando dados/ e src/.")

ROOT = achar_raiz()
sys.path.insert(0, str(ROOT / "src"))
print("Repositório:", ROOT)


## Ablar o tamanho de chunks

In [ ]:
from iad.retrieval import load_corpus,chunk_headings,TfidfIndex
from iad.metrics import recall_at_k
base=load_corpus(ROOT/'dados/relatorios');base.update(load_corpus(ROOT/'dados/faq'));base.update(load_corpus(ROOT/'dados/comunicados'))
q=pd.read_csv(ROOT/'dados/perguntas_rag.csv')
gold=[None if x=='nenhuma' else x for x in q.documento_esperado]
for max_chars in [250,650,1300]:
    chunks=chunk_headings(base,max_chars=max_chars)
    idx=TfidfIndex(chunks)
    ranks=[[h.doc_id for h in idx.search(texto,k=5)] for texto in q.pergunta]
    print(max_chars,'chars, chunks:',len(chunks),'Recall@2:',round(recall_at_k(ranks,gold,2),3))

## Mudar k no mesmo experimento

In [ ]:
idx=TfidfIndex(chunk_headings(base,max_chars=650))
ranks=[[h.doc_id for h in idx.search(texto,k=10)] for texto in q.pergunta]
print({k:recall_at_k(ranks,gold,k) for k in [1,2,3,5]})
print('Recall retrieval não equivale a fidelidade de geração.')

## Síntese individual

1. Qual pergunta este experimento responde e qual **não** responde?
2. Que valores foram calculados e quais foram apenas interpretados?
3. Que alteração você realizou? Registre a saída e explique sua escolha.
4. Qual erro seria especialmente relevante para uma decisão em **Macroeconomia e Finanças**?

**Entrega:** Comparação controlada de três variantes de RAG com relatório de erros.